In [2]:
import gymnasium
import optuna
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.envs.registration import register


# PPO

In [ ]:
def objective(trial):
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
    n_steps = trial.suggest_int('n_steps', 512, 4096)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])
    n_epochs = trial.suggest_int('n_epochs', 3, 15)
    gamma = trial.suggest_uniform('gamma', 0.9, 0.999)
    gae_lambda = trial.suggest_uniform('gae_lambda', 0.8, 1.0)
    clip_range = trial.suggest_uniform('clip_range', 0.1, 0.3)
    clip_range_vf = trial.suggest_uniform('clip_range_vf', 0.1, 0.4) if trial.suggest_categorical('use_clip_range_vf', [True, False]) else None
    ent_coef = trial.suggest_uniform('ent_coef', 0.0, 0.01)
    vf_coef = trial.suggest_uniform('vf_coef', 0.1, 1.0)
    max_grad_norm = trial.suggest_uniform('max_grad_norm', 0.3, 1.0)
    target_kl = trial.suggest_uniform('target_kl', 0.01, 0.1) if trial.suggest_categorical('use_target_kl', [True, False]) else None

    register(
    id="reactor_v2",
    entry_point="reactor_env:Reactor",
)
    # Create the environment
    env = gymnasium.make('reactor_v2')  # Replace with your specific environment

    # Create the PPO model with the suggested hyperparameters
    model = PPO(
        'MlpPolicy',
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=gamma,
        gae_lambda=gae_lambda,
        clip_range=clip_range,
        clip_range_vf=clip_range_vf,
        ent_coef=ent_coef,
        vf_coef=vf_coef,
        max_grad_norm=max_grad_norm,
        target_kl=target_kl
    )

    # Train the model
    model.learn(total_timesteps=300_000)  # Adjust the number of timesteps as needed

    # Evaluate the model
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=10)

    return mean_reward  # We want to maximize the mean reward


In [ ]:
# Create a study object and optimize
study = optuna.create_study(direction='maximize')  # We want to maximize the mean reward
study.optimize(objective, n_trials=10)  # Set the number of trials you want


[I 2024-11-11 13:03:29,644] A new study created in memory with name: no-name-97908c94-ce8c-47cf-b92c-8cc1dad886a0
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_7336\4089008711.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_7336\4089008711.py:6: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  gamma = trial.suggest_uniform('gamma', 0.9, 0.999)
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_7336\4089008711.py:7: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v

In [5]:
# Print the best hyperparameters
best_params = study.best_params

for key,value in best_params.items():
    print(f"{key} : {value}")
    

learning_rate : 0.00034996020317115933
n_steps : 3407
batch_size : 128
n_epochs : 4
gamma : 0.9608497746183995
gae_lambda : 0.9185758625807959
clip_range : 0.22653261622750495
use_clip_range_vf : True
clip_range_vf : 0.3671280814187559
ent_coef : 0.00778656556107332
vf_coef : 0.9641662335931005
max_grad_norm : 0.8408041712341976
use_target_kl : True
target_kl : 0.06300059841587244


# SAC

In [3]:
def objective(trial):
    # Hyperparameter suggestions
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
    buffer_size = trial.suggest_int('buffer_size', int(1e4), int(1e6))
    learning_starts = trial.suggest_int('learning_starts', 100, 10000)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])
    tau = trial.suggest_uniform('tau', 0.001, 0.05)
    gamma = trial.suggest_uniform('gamma', 0.9, 0.999)
    train_freq = trial.suggest_int('train_freq', 1, 10)
    gradient_steps = trial.suggest_int('gradient_steps', 1, 30)
    ent_coef = trial.suggest_loguniform('ent_coef', 0.0001, 0.1) if trial.suggest_categorical('use_ent_coef_auto', [False, True]) else 'auto'

    # Register and create the environment
    register(
        id="reactor_v2",
        entry_point="reactor_env:Reactor",
    )
    env = gymnasium.make('reactor_v2')  # Replace with your specific environment

    # `target_entropy` calculation depends on `env`
    target_entropy = trial.suggest_uniform('target_entropy', -env.action_space.shape[0], 0) if trial.suggest_categorical('use_target_entropy_auto', [False, True]) else 'auto'

    # Model initialization
    model = SAC(
        'MlpPolicy',
        env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        learning_starts=learning_starts,
        batch_size=batch_size,
        tau=tau,
        gamma=gamma,
        train_freq=train_freq,
        gradient_steps=gradient_steps,
        ent_coef=ent_coef,
        target_update_interval=trial.suggest_int('target_update_interval', 1, 100),
        target_entropy=target_entropy
    )

    # Train the model
    model.learn(total_timesteps=500_000)  # Adjust the number of timesteps as needed

    # Evaluate the model
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=10)

    return mean_reward  # We want to maximize the mean reward


# Create a study object and optimize
study = optuna.create_study(direction='maximize')  # We want to maximize the mean reward
study.optimize(objective, n_trials=10)  # Set the number of trials you want

# Print the best hyperparameters
best_params = study.best_params

for key, value in best_params.items():
    print(f"{key} : {value}")


[I 2024-12-28 13:25:43,394] A new study created in memory with name: no-name-3b9be366-34fe-4f43-9198-1ea23cd40597
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_17756\1290588729.py:3: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-3)
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_17756\1290588729.py:7: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  tau = trial.suggest_uniform('tau', 0.001, 0.05)
C:\Users\Reuel Group\AppData\Local\Temp\ipykernel_17756\1290588729.py:8: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v

learning_rate : 2.0100866791609192e-05
buffer_size : 127415
learning_starts : 8942
batch_size : 512
tau : 0.0265783511470118
gamma : 0.920548515347066
train_freq : 1
gradient_steps : 20
use_ent_coef_auto : False
use_target_entropy_auto : True
target_entropy : -0.7978268822546424
target_update_interval : 16


In [4]:
best_params

{'learning_rate': 2.0100866791609192e-05,
 'buffer_size': 127415,
 'learning_starts': 8942,
 'batch_size': 512,
 'tau': 0.0265783511470118,
 'gamma': 0.920548515347066,
 'train_freq': 1,
 'gradient_steps': 20,
 'use_ent_coef_auto': False,
 'use_target_entropy_auto': True,
 'target_entropy': -0.7978268822546424,
 'target_update_interval': 16}